# Chapter 10. 언어 모델을 위한 신경망
- 순환 신경망의 한계를 뛰어넘기 위해 고안된 어텐션 메커니즘과 트랜스포머 구조 학습
- 트랜스포머 모델의 구성 방식에 대해 배우고, 인코더와 디코더 기반 모델의 차이 화깅ㄴ
- 사전 훈련된 트랜스포머 기반의 언어 모델로 텍스트를 요약하거나 입력된 프롬프트를 
기반으로 새로운 텍스트를 생성하는 방법을 학습

## Ch 10-1 어텐션 메커니즘과 트랜스포머

- 순환 신경망은 텍스트와 같은 시퀀스 데이터는 시퀀스가 길어질수록 이전에 처리한 데이터가 희석됨
- 이 문제를 해결하기 위해 LSTM과 GRU 같은 구조가 개발되었지만, 완벽한 해결책이 되지 못함
- 전체 트랜스포머 모델

<center><img src = 'image-8.png'></center>

### 순환 신경망을 사용한 인코더-디코더 네트워크
- 자기회귀 모델 autoregressive model

- 인코더 신경망은 입력된 문장을 단어 (토큰) 단위로 하나씩 처리하면서 전체 정보를 하나의 은닉 상태에 압축
- 그런 다음, 디코더 신경망이 이 은닉 상태를 받아 마찬가지로 한 단어씩 번역된 문장을 생성

<center><img src = 'image.png' > </center>

(인코더에 입력되는 텍스트와 디코더에서 출력되는 텍스트의 길이는 그림과 다르게 다를 수 있음)
- 단어 (토큰) 단위로 하나씩 처리하면서 전체 정보를 하나의 은닉 상태에 압축 
- 그런 다음, 디코더 신경망이 이 은닉 상태를 받아 마찬가지로 한 단어씩 번역된 문장을 생성
- 번역할 문장이 길어질수록 초기에 입력된 내용을 기억하기 어려워지고, 특히, 디코더 신경망은 인코더의 마지막 은닉 상태만 참고하여 번역을 수행하기 때문에 이런 문제가 더 심해짐

- 인코더와 디코더는 텍스트를 한 토큰씩 처리
- 입력할 때도 한 토큰씩 받고, 출력할 때도 한 토큰씩 생성해서 속도가 느림 
    - 디코더는 T라는 토큰을 생성한 후, 인코더의 마지막 은닉 상태와 자신의 은닉 상태를 활용해 "love"를 만듦
    - "I love"를 생성한 후 같은방식으로 인코더의 마지막 은닉 상태와 자신의 은닉 상태를 활용해 “you”를 출력

### 어텐션 메커니즘 attention mechanism
- 디코더가 인코더의 마지막 은닉 상태만 참고하여 번역을 수행했지만, 어텐션 메커니즘을 사용하면 인코더의 모든 타임스텝에서 계산된 은닉 상태를 활용

<center><img src = 'image-1.png'></center>

- 디코더가 두 번째 타임스텝에서 “love”라는 토큰을 생성할 때, 인코더의 모든 타임스텝에서 출력된 모든 은닉 상태를 참고

#### 어텐션 가중치
- 다른 모델 파라미터와 같이 신경망을 훈련하면서 함께 학습
- 디코더가 인코더의 은닉 상태를 활용하는 방식은 가중치를 곱하는 형태로 이루어짐 
- 디코더는 인코더의 모든 타임스텝에서 생성된 은닉 상태를 동일하게 참고하는 것이 아니라, **각 은닉 상태마다 가중치를 다르게 적용**하여 더 중요한 정보를 강조
- 그림 속 $a_1, a_2, a_3$ 해당
    - 값들은 각각 다른 값을 가지며, 디코더의 타임스텝마다 달라질 수 있음

> - 장점: 긴 텍스트를 처리할 때 정보 손실을 줄이는 데 매우 효과적

> - 단점: 어텐션 가중치를 계산하기 위해 인코더의 모든 타임스텝에서 생성된 은닉 상태를 저장해야 하므로 연산량이 증가
> - 인코더가 처리할 수 있는 타임스텝의 최대 개수를 정해야 하며, 이로 인해 입력 텍스트의 길이가 제한될 수 있음 
> - 또한, 어텐션을 사용해도 여전히 한 번에 한 토큰씩 처리해야 함

### **트랜스포머 Transformer**
- Attention Is All You Need 논문 참고
- 어텐션 메커니즘을 다양한 기술을 조합하여 사용
- 기존의 인코더-디코더 구조를 유지하면서도 순환 신경망을 완전히 제거
    - 입력 텍스트를 한 토큰씩 처리할 필요 없이 한 번에 모두 처리

<center><img src = image-2.png></center>

#### 작동 방식
- 입력 텍스트를 한꺼번에 처리하기 때문에  타입 스텝 개념이 필요하지 않음

> - 장점 : 한 번에 모두 처리할수 있어 모델의 처리 속도가 크게 향상
> - 단점 : 입력 텍스트의 길이에 제한이 있음 

- 인코더에서 처리된 결과는 디코더에 전달되며, 디코더는 이를 바탕으로 번역된 문장을 생성

은닉 벡터hidden vector = 단어 벡터word vector = 임베딩 벡터 embedding vector

디코더는 인코더에서 전달받은 은닉 벡터를 활용해 각 타임스텝에서 출력할 토큰을 생성
인코더-디코더 모델처럼, 디코더는 이전에 생성된 토큰을 참고하면서 새로운 토큰을 생성
순환 신경망 없이도 이전 출력값을 반영할 수 있는 구조를 갖추고 있음

#### 셀프 어텐션 메커니즘
- 셀프 어텐션 self attention : 인코더에 입력되는 토큰만으로 어텐션 가중치를 학습
- 어텐션 헤드attentton head : 셀프 어텐션 연산을 수행하는 하나의 단위(위의 1.부터 6.까지)

##### 하나의 어텐션 헤드의 연산 과정
1) 전체 토큰이 단어 임베딩을 거친 후 한 번에 밀집층에 통과
2) 첫 번째 밀집층을 통과한 벡터(쿼리 Query 벡터로 지칭)는 두 번째 밀집층에 통과
3) 두 번째 밀집층에 통과한 벡터(키 key 벡터로 지칭)가 만들어짐
4) 쿼리 벡터와 키 벡터를 곱하여 어텐션 점수(attnetion score)가 계산하여 어텐션 행렬(attention matrix)를 생성
    - 벡터 곱에 따라 쿼리 벡터가 3개 키 벡터가 3개인 경우 총 9개의 어텐션 점수가 나옴
5) 입력 텍스트를 또다른 밀집층에 통과시켜 값 value 벡터를 계산
6) 계산된 어텐션 점수를 값 벡터에 곱해서 최종적인 셀프 어텐션 출력을 생성
    - 어텐션 출력 벡터는 각 입력 토큰이 다른 토큰들과 얼마나 관련이 있는지를 반영한 은닉 벡터라고 할 수 있음
    -  모델은 문맥을 더 정확하게 이해하고, 중요한 정보를 효과적으로 강조할 수 있음

- 쿼리, 키, 값 벡터를 생성하는 밀집층 세 개의 가중치도 함께 학습

<center><img src = 'image-3.png'></center>



- 멀티 헤드 어턴션multi-head attentton : 트랜스포머에에서 사용하는 방식으로 여러개의 어텐션 헤드를 사용

<center><img src = 'image-4.png'></center>

- 각 어텐션 헤드에서는 쿼리, 키, 값 벡터를 생성하는 밀집층이 서로 다르게 사용
- 어텐션 헤드들의 출력은 하나로 합쳐진 후, 밀집층을 통과하여 어텐션 층의 최종 출력이 됩니다. 헤드의 개수는 모델마다 다르며, 보통 몇 개에서 많게는 수십 개까지 사용

### 정규화
- 딥러닝에서는 여러 개의 층을 거치면서 특성의 스케일이 변할 수 있기 때문에, 단순한 입력 정규화만으로는 충분하지 않음

#### 배치 정규화batch normalization
- 층과 층 사이에 놓이며, 이전 층의 출력을 배치 단위로 평균과 분산을 계산하여 평균이 0, 분산이 1 이 되도록 조정한 후, 다음 층으로 전달
- 모든 특성을 동일한 분포（평균 0, 분산 1）로 정규화하면 신경망이 학습한 유용한 정보가 손실될 수 있습니다. 이를 방지하기 위해 배치 정규화 층은 평균과 분산의 양을 조정하는 두 개의 파라미터를 학습하여 정규화를 수행

<center><img src = 'image-5.png'></center>

- 파란 색으로 표시된 부분이 정규화가 적용되는 단위로, 모든 샘플에서 특정 채널의 데이터를 모아 평균과 분산을 계산한 후 정규화를 적용 즉 배치 단위로 정규화를 수행
- 배치 정규화를 적용하면 훈련 속도가 빨라지고, 학습 과정이 안정화되나 텍스트 데이터는 샘플 길이가 제각각이라 적용이 어려움

#### 층 정규화
- 텍스트 모델에서 쓰이는 정규화
- 층 정규화는 각 샘플의 토큰마다 개별적으로 정규화를 수행.

<center><img src = 'image-6.png'></center>

- 주로 멀티 헤드 어텐션 층 다음에 드롭아웃과 층 정규화가 사용
- 잔차 연결 residual connectionon aka 스킵 연결 skip connection : 층이 많을수록 훈련이 어려워 지는데,  멀티 헤드 어텐션 층을 거친 출력에 입력값을 그대로 더함
- 신경망이 훈련될 때는 뒤에서부터 거꾸로 모델의 파라미터 업데이트 신호가 전파
- 잔차 연결이 추가되면 이 신호가 멀티 헤드 어텐션 층을 거치지 않고 직접 앞쪽 층으로 전달하여 신경망 층이 쌓여도 효과적으로 훈련 할 수 있음
- 트랜스포머의 인코더 역시 두 개의 잔차 연결을 사용하며, 두 번째 잔차 연결은 다음에 배울 피드포워드 네트워크의 앞뒤를 연결하는 역할을 함

### 피드포워드 네트워크와 인코더 블록
#### 피드포워드 네트워크 feed forward network
- 일반적인 피드포워드 신경망을 의미하는 것이 아님
- 트랜스포머의 인코더에서 멀티 헤드 어텐션과 층 정규화 다음에 나오는 밀집층을 지칭

- 두 개의 밀집층으로 구성
    - 첫 번째 밀집층은 ReLU 활성화 함수 
    - 두 번째 밀집층은 활성화 함수를 사용하지 않음 
- 그 다음 다시 드롭아웃 층이 추가되며, 이 세 개의 층을 또 다른 잔차 연결이 감싸게 됨

<center><img srg = 'image-7.png'></center>

In [1]:
from transformers.utils import logging
from transformers import pipeline
from transformers import AutoTokenizer

2025-12-03 21:37:24.409248: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-03 21:37:24.946703: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 21:37:27.691331: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [25]:
import numpy as np
from scipy.special import softmax
from openai import OpenAI

In [39]:
from dotenv import load_dotenv
import os
load_dotenv()

True

## Ch 10-2

In [2]:
pipe = pipeline(task='summarization', device=0)

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


In [4]:
pipe = pipeline(task='summarization',
                model='sshleifer/distilbart-cnn-12-6', device=0)

Device set to use cuda:0


In [5]:
sample_text = """Vincent Willem van Gogh was a Dutch Post-Impressionist painter who is among the most famous and influential figures in the history of Western art. In just over a decade, he created approximately 2100 artworks, including around 860 oil paintings, most of them in the last two years of his life. His oeuvre includes landscapes, still lifes, portraits, and self-portraits, most of which are characterised by bold colours and dramatic brushwork that contributed to the rise of expressionism in modern art. Van Gogh's work was beginning to gain critical attention before he died from a self-inflicted gunshot at age 37. During his lifetime, only one of Van Gogh's paintings, The Red Vineyard, was sold.
"""
pipe(sample_text)

[{'summary_text': " Vincent Willem van Gogh was a Dutch Post-Impressionist painter . His oeuvre includes landscapes, still lifes, portraits and self-portraits . Van Gogh's work was beginning to gain critical attention before he died from a self-inflicted gunshot at age 37 ."}]

In [6]:
kobart = pipeline(task='summarization',
                  model='EbanLee/kobart-summary-v3', device=0)

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
Device set to use cuda:0


In [7]:
ko_text = """하나, ‘입문자 맞춤형 7단계 구성’을 따라가며 체계적으로 반복하는 탄탄한 학습 설계!
이 책은 데이터 분석의 핵심 내용을 7단계에 걸쳐 반복 학습하면서 자연스럽게 머릿속에 기억되도록 구성했습니다. [핵심 키워드]와 [시작하기 전에]에서 각 절의 주제에 대한 대표 개념을 워밍업하고, 이론과 실습을 거쳐 마무리에서는 [핵심 포인트]와 [확인 문제]로 한번에 복습합니다. ‘혼자 공부할 수 있는’ 커리큘럼을 그대로 믿고 끝까지 따라가다 보면 데이터 분석 공부가 난생 처음인 입문자도 무리 없이 책을 끝까지 마칠 수 있습니다!
둘, 실제로 일어날 법한 흥미로운 스토리에 담긴 문제를 직접 해결하며 익히는 ‘진짜’ 데이터 분석!
현장감 넘치는 스토리를 통해 데이터를 다루는 방법을 알려 주어 ‘파이썬’과 ‘데이터’가 낯설어도 몰입감 있는 학습을 할 수 있도록 구성했습니다. 이 책에서는 API와 웹 스크래핑을 통해 실제 도서관 데이터와 온라인 서점 웹사이트에서 데이터를 가져오는 등 내 주변에 있는 데이터를 직접 수집할 수 있는 방법을 가이드합니다. 또한 판다스, 넘파이, 맷플롯립 등 데이터 분석에 유용한 각종 파이썬 라이브러리를 활용해 보며 코딩 감각을 익히고, 핵심 통계 지식으로 기본기를 탄탄하게 다질 수 있습니다. 마지막에는 분석을 바탕으로 미래를 예측하는 머신러닝까지 맛볼 수 있어 데이터 분석의 처음부터 끝까지 제대로 배울 수 있습니다.
셋, ‘혼공’의 힘을 실어줄 동영상 강의와 혼공 학습 사이트 지원!
책으로만 학습하기엔 여전히 어려운 입문자를 위해 저자 직강 동영상도 지원합니다. 또한 학습을 하며 궁금한 사항은 언제든지 저자에게 질문할 수 있도록 학습 사이트를 제공합니다. 저자가 질문 하나하나에 직접 답변을 달아 주는 것은 물론, 관련 최신 기술과 정보도 얻을 수 있습니다. 게다가 혼자 공부하고 싶지만 정작 혼자서는 자신 없는 사람들을 위해 혼공 학습단을 운영합니다. 혼공 학습단과 함께하면 마지막까지 포기하지 않고 완주할 수 있을 것입니다.
▶ https://hongong.hanbit.co.kr
▶ https://github.com/rickiepark/hg-da
넷, 언제 어디서든 가볍게 볼 수 있는 혼공 필수 [용어 노트] 제공!
꼭 기억해야 할 핵심 개념과 용어만 따로 정리한 [용어 노트]를 제공합니다. 처음 공부하는 사람들이 프로그래밍을 어려워하는 이유는 낯선 용어 때문입니다. 그러나 어려운 것이 아니라 익숙하지 않아서 헷갈리는 것이므로, 용어나 개념이 잘 생각나지 않을 때는 언제든 부담 없이 [용어 노트]를 펼쳐 보세요. 제시된 용어 외에도 새로운 용어를 추가하면서 자신만의 용어 노트를 완성해가는 과정도 또 다른 재미가 될 것입니다.
"""

kobart(ko_text)

[{'summary_text': '이 책은 데이터 분석의 핵심 내용을 7단계에 걸쳐 반복 학습하면서 머릿속에 기억되도록 구성했습니다. 독자 공부할 수 있는 커리큘럼을 그대로 믿고 끝까지 따라가다 보면 데이터 분석 공부가 난생 처음인 입문자도 무리 없이 책을 끝까지 마칠 수 있습니다. 현장감 넘치는 스토리를 통해 데이터를 다루는 방법을 알려 주어 몰입감 있는 학습을 할 수 있도록 구성했습니다. 저자가 질문 하나하나에 직접 답변을 달아 주는 것은 물론, 최신 기술과 정보도 얻을 수 있습니다. 혼공 학습단과 함께하면 마지막까지 포기하지 않고 완주할 수 있을 것입니다. 꼭 기억해야 할 핵심 개념과 용어만 따로 정리한 [용어 노트]를 제공합니다. 새로운 용어를 추가하면서 자신만의 용어 노트를 완성해가는 과정도 재미가 될 것입니다.'}]

In [8]:
print(kobart.model.config)

BartConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "author": "EbanLee(rudwo6769@gmail.com)",
  "bos_token_id": 1,
  "classif_dropout": 0.1,
  "classifier_dropout": 0.1,
  "d_model": 768,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 3072,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 1,
  "do_blenderbot_90_layernorm": false,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 3072,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 1,
  "extra_pos_embeddings": 2,
  "force_bos_token_to_be_generated": false,
  "forced_eos_token_id": 1,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "kobart_version": 2.0,
  "label2id": {
    "NEGATIVE": 0

In [9]:
kobart.model

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(30000, 768, padding_idx=3)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(30000, 768, padding_idx=3)
      (embed_positions): BartLearnedPositionalEmbedding(1028, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_lay

### 텍스트 토큰화

In [10]:
print(kobart.tokenizer.vocab_size)

30000


In [11]:
len(kobart.tokenizer)

30000

In [12]:
vocab = kobart.tokenizer.vocab
len(vocab)

30000

In [14]:
list(vocab.items())[:10]

[('▁신기', 25281),
 ('▁하며,', 20991),
 ('부(', 18587),
 ('▁“아', 22284),
 ('▁있었다.', 17626),
 ('▁여객', 22301),
 ('랍', 10224),
 ('▁밤에', 26477),
 ('▁한다"고', 16736),
 ('爵', 5555)]

In [15]:
tokens = kobart.tokenizer.tokenize('혼자 만들면서 배우는 딥러닝')
print(tokens)

['▁혼자', '▁만들', '면서', '▁배우는', '▁', '딥', '러', '닝']


In [16]:
kobart.tokenizer.convert_tokens_to_ids(tokens)

[16814, 14397, 14125, 25429, 1700, 10021, 10277, 9747]

In [17]:
token_ids = kobart.tokenizer.encode('혼자 만들면서 배우는 딥러닝')
print(token_ids)

[0, 16814, 14397, 14125, 25429, 1700, 10021, 10277, 9747, 1]


In [18]:
tokens = kobart.tokenizer.convert_ids_to_tokens(token_ids)
print(tokens)

['<s>', '▁혼자', '▁만들', '면서', '▁배우는', '▁', '딥', '러', '닝', '</s>']


In [19]:
kobart.tokenizer.decode(token_ids)

'<s> 혼자 만들면서 배우는 딥러닝</s>'

## Ch 10-3 대규모 언어 모델로 텍스트 생성하기

### EXAONE-3.5로 상품 질문에 대한 대답 생성하기

In [20]:
exaone_tokenizer = AutoTokenizer.from_pretrained(
    "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")

In [21]:
pipe = pipeline(task="text-generation",
                model="LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
                tokenizer=exaone_tokenizer,
                device=0, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [22]:
messages = [
    {"role": "system",
     "content": "너는 쇼핑몰 홈페이지에 올라온 질문에 대답하는 Q&A 챗봇이야. \
                 확정적인 답변을 하지 말고 제품 담당자가 정확한 답변을 하기 위해 \
                 시간이 필요하다는 간단하고 친절한 답변을 생성해줘."},
    {"role": "user", "content": "이 다이어리에 내년도 공휴일이 표시되어 있나요?"}
]

pipe(messages, max_new_tokens=200)

[{'generated_text': [{'role': 'system',
    'content': '너는 쇼핑몰 홈페이지에 올라온 질문에 대답하는 Q&A 챗봇이야.                  확정적인 답변을 하지 말고 제품 담당자가 정확한 답변을 하기 위해                  시간이 필요하다는 간단하고 친절한 답변을 생성해줘.'},
   {'role': 'user', 'content': '이 다이어리에 내년도 공휴일이 표시되어 있나요?'},
   {'role': 'assistant',
    'content': '안녕하세요! 다이어리에 내년의 공휴일 정보가 표시되어 있는지에 대해 정확하게 답변 드리기 위해서는 제품 담당자분께 확인을 받아야 할 것 같아요. 현재로선 공식적인 답변이 어려우니, 가능하시다면 고객센터나 직접 저희 온라인 스토어에 연락해 주시면 더 정확한 정보를 제공해드릴 수 있을 것 같습니다. 감사합니다!'}]}]

In [ ]:
pipe(messages, max_new_tokens=200, return_full_text=False)

[{'generated_text': '네, 궁금하시군요! 저희가 확인해보니 다이어리에 정확한 공휴일 정보가 미리 표시되어 있지는 않아요. 하지만, 가장 확실한 정보를 얻으시려면 저희 고객센터에 문의하시거나, 공휴일 정보가 업데이트된 공식 달력이나 정부 웹사이트를 확인하시는 게 좋을 것 같아요. 시간 내주셔서 감사합니다! 빠르게 답변드리도록 할게요. 😊'}]

In [24]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True)
print(output[0]['generated_text'])

죄송하지만, 저는 실시간 정보에 접근할 수 없어서 다이어리에 내년의 공휴일이 정확히 표시되어 있는지 확인해 드릴 수 없습니다. 정확한 내용을 확인하려면 제품 담당자나 직접 판매 사이트의 고객 서비스에 연락하시는 게 좋을 것 같습니다. 그들이 가장 최신의 정보를 제공해 주실 거예요! 추가로 궁금한 점이 있으시면 알려주세요.


### 토큰 디코딩 전략

#### 기본 샘플링

In [26]:
logits = np.array([1, 2, 3, 4, 100])

In [27]:
probas = softmax(logits)
print(probas)

[1.01122149e-43 2.74878501e-43 7.47197234e-43 2.03109266e-42
 1.00000000e+00]


In [28]:
np.random.multinomial(100, probas)

array([  0,   0,   0,   0, 100])

In [29]:
probas = softmax(logits/100)
np.random.multinomial(100, probas)

array([11, 19, 19, 15, 36])

In [30]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=10.0)
print(output[0]['generated_text'])

맞셔봐염~~ 🗐 honestly sir./sshir!! For an optimal answer straight for customer service folks back to ye regarding such intricate date inquiries specific only on every 다이어리 상품 line, maybe givethem afixed amountoffastby. Thebestexperienceistoreacharepresentantto clarifyimpediumpointslikewhat year it refers too - iscountingthestandingcontentfromnowforthneyaarapriarilybyyearannsoundconcesssationalwayoryouwaitawaitmaybinthememailresponsewithadefinityimpecculateintentthusforwardbyapowerpackforinformationgoesstrously? Probably needawaterlooemofmoment! 😩 💓 Just let a few customer servicedekshonekodheescheasethatfreeduebegainsecuredcallingandthemanyou’llwant! ☕ 😉 😉❄


In [31]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=0.001)
print(output[0]['generated_text'])

안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 정확한 답변을 드리기 위해서는 제품 담당자에게 확인이 필요합니다. 현재로선 직접 확인이 어려우니, 저희가 안내드릴 수 있는 방법으로는 고객센터에 연락하시거나, 제품 페이지 내의 문의 게시판을 통해 질문해 보시는 것이 좋을 것 같습니다. 담당자분께서 빠르게 답변해 주실 거예요! 감사합니다.


#### top-k 샘플링

In [32]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_k=10)
print(output[0]['generated_text'])

안녕하세요! 다이어리에 내년도 공휴일 정보가 포함되어 있는지 확인해드리려면, 제품 담당자께 직접 문의하시는 게 가장 정확할 것 같아요. 그분께서는 해당 제품의 최신 정보와 상세한 특징들을 자세히 알려주실 수 있을 거예요. 혹시 곧 연락이 가능하시다면, 거기에 문의하시는 것을 추천드립니다! 감사합니다.


In [33]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_k=10, temperature=10.0)
print(output[0]['generated_text'])

제가 즉시 답변드린다며 다이어리 안에 내년에 해당하는 공휴일 정보 표시 여부 확인 방법은 어렵기 때문에 직접 상품 페이지 확인을 recommends드려 드릴게요._productPage_ 혹은_문의하기 메뉴로 연결하시길 바랍니다._That allows our staff to check directly based solely off the latest release info!  빠른 도움 주시길 기다릴거에뇨! 📝🗳錄  더 구체한 확인이 필요하면 고객서비스에 연락해 보시구요 😜🏃‍♀😊 🕯🌹 감사하세요 😇  시간 내서 직접 답변 드릴게요 🗞🧐 😖🏥🏼🏝️🏠 🙅‍🚩〉  곧 자세히 알고 드릴게요 🤙☕


#### top-p 샘플링

In [ ]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_p=0.9)
print(output[0]['generated_text'])

In [ ]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=0.8, top_k=100, top_p=0.9)
print(output[0]['generated_text'])

### GPT-4o로 상품 질문에 대한 대답 생성하기

In [43]:
API_KEY = os.environ.get('API_KEY')
client = OpenAI(api_key=API_KEY)

In [44]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print(completion.choices[0].message.content)

In [ ]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    top_p=0.9
)
print(completion.choices[0].message.content)

In [ ]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=1.8
)
print(completion.choices[0].message.content)